In [1]:
"""
Walks a local directory of downloaded SEC filings structured as:
    <root>/<TICKER>/<FORM_TYPE>/<DATE>/<ACCESSION_NUMBER>/*.htm

e.g. D:\\Fin_Rag\\sec_filings\\AAPL\\10-K\\2020-09-26\\0000320193-20-000096\\

For each accession folder, identifies the main filing document (not an
exhibit/certification file), reads its RAW HTML from disk, and runs it
through extract_reports_from_filing() -- this is genuine raw HTML, unlike
anything obtainable through a markdown-converting web fetch, so this is
the first real test of that code path.

Usage:
    python batch_ingest_filings.py "D:\\Fin_Rag\\sec_filings" --write-jsonl reports.jsonl
    # or, to ingest straight into earnings_agent's Postgres + Chroma stores:
    python batch_ingest_filings.py "D:\\Fin_Rag\\sec_filings" --ingest
"""

import argparse
import json
from pathlib import Path

from filing_extraction import extract_reports_from_filing

# Filenames matching these patterns are exhibits/certifications/graphics,
# not the main filing body -- skip them when picking the document to parse.
_EXCLUDE_PATTERNS = ["ex-", "ex_", "-ex", "_ex", "cert", "graphic", "image", "logo"]


In [2]:
#%pip install psycopg2
#%pip install lxml
#%pip install langchain_anthropic

In [3]:

def find_main_document(accession_dir: Path, ticker: str, date_str: str) -> Path | None:
    """Picks the primary filing document out of an accession folder, which
    typically also contains exhibits, XBRL instance files, and images.

    Strategy, in order:
      1. Exact match on SEC's usual naming convention: {ticker}-{yyyymmdd}.htm
      2. Any .htm/.html file NOT matching an exhibit-like pattern, preferring
         the largest by file size (the main document is almost always the
         biggest file in the folder -- exhibits are short).
    """
    htm_files = list(accession_dir.glob("*.htm")) + list(accession_dir.glob("*.html"))
    if not htm_files:
        return None

    compact_date = date_str.replace("-", "")
    expected_name = f"{ticker.lower()}-{compact_date}.htm"
    for f in htm_files:
        if f.name.lower() == expected_name:
            return f

    candidates = [f for f in htm_files if not any(p in f.name.lower() for p in _EXCLUDE_PATTERNS)]
    if not candidates:
        candidates = htm_files   # fall back to everything if the exclude filter removed all of them

    return max(candidates, key=lambda f: f.stat().st_size)


def parse_filing_path(accession_dir: Path, root: Path) -> dict:
    """Extracts ticker/form_type/date/accession from the folder structure
    itself -- this metadata is already encoded in the path, so no need to
    parse it out of the filing content."""
    rel_parts = accession_dir.relative_to(root).parts
    if len(rel_parts) < 4:
        raise ValueError(f"Unexpected path depth under root: {accession_dir}")
    ticker, form_type, date_str, accession = rel_parts[-4:]
    return {"ticker": ticker, "form_type": form_type, "date": date_str, "accession": accession}


def find_all_accession_dirs(root: Path) -> list[Path]:
    """An accession dir is any directory at exactly ticker/form/date/accession
    depth under root that contains at least one .htm/.html file."""
    accession_dirs = []
    for candidate in root.glob("*/*/*/*"):
        if candidate.is_dir() and (list(candidate.glob("*.htm")) or list(candidate.glob("*.html"))):
            accession_dirs.append(candidate)
    return accession_dirs


def read_html_file(path: Path) -> str:
    """SEC filings are usually UTF-8, but older ones can be latin-1 --
    try UTF-8 first, fall back rather than crashing the whole batch on one
    bad file."""
    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        print(f"  UTF-8 decode failed for {path.name}, retrying as latin-1")
        return path.read_text(encoding="latin-1")


def process_filing_directory(root_dir: str) -> list[dict]:
    root = Path(root_dir)
    accession_dirs = find_all_accession_dirs(root)
    print(f"Found {len(accession_dirs)} filing folders under {root}")

    all_reports = []
    for accession_dir in accession_dirs:
        try:
            meta = parse_filing_path(accession_dir, root)
        except ValueError as e:
            print(f"  Skipping {accession_dir}: {e}")
            continue

        main_doc = find_main_document(accession_dir, meta["ticker"], meta["date"])
        if main_doc is None:
            print(f"  No .htm/.html file found in {accession_dir}, skipping")
            continue

        print(f"\n{meta['ticker']} {meta['form_type']} {meta['date']} -> {main_doc.name}")
        try:
            html = read_html_file(main_doc)
        except Exception as e:
            print(f"  Failed to read {main_doc}: {e}")
            continue

        id_prefix = f"{meta['ticker']}_{meta['form_type']}_{meta['date']}_{meta['accession']}"
        try:
            reports = extract_reports_from_filing(
                html, company=meta["ticker"], fiscal_period=meta["date"],
                id_prefix=id_prefix, is_html=True,
            )
        except Exception as e:
            # A parsing failure on ONE filing shouldn't kill the whole batch --
            # log it and keep going, same philosophy as the per-table skip
            # logic inside extract_tables_from_html itself.
            print(f"  extract_reports_from_filing failed for {main_doc}: {e}")
            continue

        print(f"  Extracted {len(reports)} table(s)")
        all_reports.extend(reports)

    return all_reports




In [4]:

all_reports = process_filing_directory(r"D:\Fin_Rag\sec_filings")
print(f"\nTotal tables extracted: {len(all_reports)}")
    
   

Found 13 filing folders under D:\Fin_Rag\sec_filings

AAPL 10-K 2020-09-26 -> aapl-20200926.htm


D:\Fin_Rag\notebooks\filing_extraction.py:252: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


  Extracted 62 table(s)

AAPL 10-K 2021-09-25 -> aapl-20210925.htm


D:\Fin_Rag\notebooks\filing_extraction.py:252: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


  Extracted 51 table(s)

AAPL 10-K 2022-09-24 -> aapl-20220924.htm


...

  Extracted 73 table(s)

AMD 10-K 2024-12-28 -> amd-20241228.htm


D:\Fin_Rag\notebooks\filing_extraction.py:252: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


  Extracted 70 table(s)

AMD 10-K 2025-12-27 -> amd-20251227.htm


D:\Fin_Rag\notebooks\filing_extraction.py:252: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


  Extracted 68 table(s)

Total tables extracted: 825


In [5]:
#%pip install langgraph

In [6]:
 # Optional: write to JSONL yourself
with open("extracted_reports.jsonl", "w") as f:
    for r in all_reports:
        f.write(json.dumps(r) + "\n")



In [7]:
import json
reports = []
with open("extracted_reports.jsonl") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        reports.append(json.loads(line))


In [8]:
from earnings_agent import ingest_report, init_postgres_schema

D:\conda_envs\finder\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\conda_envs\finder\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [10]:

# Each entry: (thread_id, question). Queries sharing a thread_id are run
# as ONE conversation -- the agent should resolve "what about 2016?" using
# context from the question before it, via InMemorySaver.
EXAMPLE_QUERIES = [
    # --- single-fact lookups ---
    ("facts", "What was Apple's net income in FY2023?"),
    ("facts", "What was AMD's revenue in its most recent fiscal year?"),
    ("facts", "What was Apple's gross margin percentage in FY2023?"),
 
    # --- trend / growth over time (needs multiple years ingested for that ticker) ---
    ("trend", "Is AMD's R&D spending increasing or decreasing?"),
    ("trend", "How has Apple's total revenue changed over the available years?"),
 
    # --- within-company comparison ---
    ("compare", "What's the difference between Apple's Products and Services revenue in FY2023?"),
    ("compare", "Which was higher for Apple in FY2023: Products revenue or Services revenue?"),
]

In [11]:

init_postgres_schema()

# Example ingestion -- replace with your actual FinQA/Kaggle-sourced reports
example_report = {
    "id": "AAPL_2017_10K",
    "company": "Apple",
    "fiscal_period": "FY2017",
    "pre_text": ["the following table summarizes selected financial data ."],
    "post_text": ["all figures in millions of dollars ."],
    "table": [
        ["", "2016", "2017"],
        ["net income", "45687", "48351"],
        ["total revenue", "215639", "229234"],
    ],
}

for r in reports:
    ingest_report(r)
print("Ingested example report.")


  

Database 'portfolio_agent' already exists -- skipping.
[ingest_report] START id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_5
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_5 built raw_text (1397 chars, has_table=True)
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_5 connecting to Postgres ...
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_5 Postgres connected, executing INSERT ...
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_5 Postgres INSERT committed
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_5 getting embedder (cached after first call) ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3185.00it/s]


[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_5 encoding text ...
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_5 embedding done (dim=384)
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_5 connecting to Chroma at ./earnings_chroma ...
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_5 upserting into Chroma collection 'earnings_reports' ...
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_5 DONE (Postgres + Chroma both written)
[ingest_report] START id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_7
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_7 built raw_text (3287 chars, has_table=True)
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_7 connecting to Postgres ...
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_7 Postgres connected, executing INSERT ...
[ingest_report] id=AAPL_10-K_2020-09-26_0000320193-20-000096_table_7 Postgres 

In [20]:
# ---------------------------------------------------------------------------
# LangGraph agent
# ---------------------------------------------------------------------------
from dotenv import load_dotenv
import os
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent
from earnings_agent import answer_earnings_query, SYSTEM_PROMPT

In [23]:


SYSTEM_PROMPT = (
    "You are a financial research assistant with access to stored earnings reports. "
    "Always call answer_earnings_query for questions about specific figures, comparisons, "
    "or trends -- do not answer from memory. Relay the tool's answer clearly, and mention "
    "the source report. If the tool cannot find a confident answer, say so explicitly."
)


load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI
LLM_MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=LLM_MODEL)

checkpointer = InMemorySaver()
earnings_agent = create_react_agent(model=llm, tools=[answer_earnings_query],
                                     prompt=SYSTEM_PROMPT, checkpointer=checkpointer)


def chat(message: str, thread_id: str = "default") -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = earnings_agent.invoke({"messages": [{"role": "user", "content": message}]}, config=config)
    return result["messages"][-1].content

 
print("\nEarnings report assistant. Type 'quit' to exit.")

thread_id = "cli-session"
for thread_id, question in EXAMPLE_QUERIES:
    print(f"\n[{thread_id}] You: {question}")
    try:
        answer = chat(question, thread_id=thread_id)
    except Exception as e:
        answer = f"(error: {e})"
    print(f"[{thread_id}] Agent: {answer}")


C:\Users\radha\AppData\Local\Temp\ipykernel_18724\2478827225.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  earnings_agent = create_react_agent(model=llm, tools=[answer_earnings_query],



Earnings report assistant. Type 'quit' to exit.

[facts] You: What was Apple's net income in FY2023?
[facts] Agent: (error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}})

[facts] You: What was AMD's revenue in its most recent fiscal year?
[facts] Agent: (error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}})

[facts] You: What was Apple's gross margin percentage in FY2023?
[facts] Agent: (error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'ty

KeyboardInterrupt: 

In [16]:
%pip install langchain-openai

Note: you may need to restart the kernel to use updated packages.
